In [2]:
import numpy as np
import polars as pl # 高速DataFrame库，用于数据处理与分析，速度更快，内存占用更低
from gensim.test.utils import common_texts
from gensim.models import Word2Vec
import pandas as pd

In [3]:
train=pl.read_parquet('/home/mingyu/Recommand-System/项目/OTTO/data/processData/train.parquet')
test=pl.read_parquet('/home/mingyu/Recommand-System/项目/OTTO/data/processData/test.parquet')

In [5]:
# pl.concat纵向合并，groupby按照session分组，agg聚合函数(不保留原维度，合并session)，pl.col选择列，alias重命名
# 为每个session合并aid组合成句子
sentences_df=pl.concat([train,test]).group_by('session').agg(pl.col('aid').alias('sentence'))

In [6]:
sentences_df

session,sentence
i32,list[i32]
4568152,"[1534113, 1534113]"
4823356,"[369910, 1004138, … 1004138]"
4496188,"[335674, 805787, … 1603001]"
10521272,"[1840615, 152602, … 1840615]"
3248044,"[951559, 847534, … 951559]"
…,…
2162153,"[327700, 579711, … 689660]"
812025,"[887053, 345153, … 1378614]"
6486998,"[996462, 1477994, … 920844]"


In [7]:
sentences=sentences_df['sentence'].to_list()

In [8]:
sentences

[[1534113, 1534113],
 [369910, 1004138, 103032, 1004138],
 [335674, 805787, 963957, 805787, 335674, 1603001],
 [1840615, 152602, 1716786, 1840615, 271193, 1840615],
 [951559, 847534, 406579, 404865, 140997, 951559, 951559, 140997, 951559],
 [1411065,
  1296995,
  602668,
  177739,
  116112,
  250729,
  1640596,
  250729,
  1640596,
  869136,
  1640596,
  250729],
 [421624],
 [905277,
  1234060,
  905277,
  905277,
  905277,
  905277,
  905277,
  1803634,
  905277,
  742814,
  1702657,
  554660,
  742814,
  122792,
  742814,
  742814,
  27530,
  27530,
  942224,
  1759412,
  551670,
  1216299,
  825401],
 [367396,
  374006,
  1485427,
  374006,
  367396,
  842590,
  923948,
  1059983,
  1383585,
  1820168,
  1399932,
  208068,
  231487,
  137514,
  368495,
  327401,
  368495,
  327401,
  368495,
  756588,
  1534335],
 [1769049,
  888530,
  850589,
  1749512,
  226439,
  1667938,
  1396230,
  951420,
  717912,
  951420,
  1281935,
  951420,
  1331376,
  951420,
  159925,
  951420,
  1396

In [9]:
import os
# 做embedding
model_path='/home/mingyu/Recommand-System/项目/OTTO/model/w2vec.model'
if os.path.exists(model_path):
    w2vec=Word2Vec.load(model_path)
else:
    # sentences训练语料，vector_size词向量维度，min_count此至少出现一次才会被训练，workers训练时使用的CPU线程数
    w2vec=Word2Vec(sentences=sentences,vector_size=32,min_count=1,workers=4)
    w2vec.save(model_path)

In [10]:
aid_list=w2vec.wv.index_to_key # 按照出现频率从高到低返回词的列表
aid_list

[1460571,
 485256,
 108125,
 29735,
 1733943,
 832192,
 184976,
 166037,
 554660,
 986164,
 231487,
 1502122,
 1603001,
 1236775,
 322370,
 332654,
 1196256,
 756588,
 959208,
 1083665,
 1022566,
 620545,
 95488,
 801774,
 247240,
 673407,
 1645990,
 1586171,
 508883,
 1116095,
 1294924,
 530377,
 811371,
 1604220,
 892871,
 152547,
 714524,
 102345,
 1531805,
 409620,
 670006,
 544144,
 1257293,
 1498443,
 1197632,
 199409,
 584027,
 819288,
 1796103,
 612920,
 1743151,
 496180,
 399315,
 636101,
 500609,
 33343,
 77440,
 1647563,
 1685214,
 632365,
 1581568,
 1462420,
 1043508,
 1182614,
 1264313,
 329725,
 861401,
 1658239,
 1497089,
 881286,
 326904,
 1111967,
 984459,
 137514,
 634452,
 794192,
 1006198,
 803928,
 1125638,
 11830,
 305158,
 1365988,
 721034,
 1406660,
 10964,
 331708,
 385065,
 1255910,
 1624436,
 1636724,
 1142000,
 190818,
 1419849,
 670066,
 159789,
 1338993,
 493104,
 884502,
 1629608,
 842590,
 450505,
 1610239,
 1052212,
 1551213,
 1383529,
 1116621,
 135997

In [12]:
# w2vec.wv[aid]可以检索到向量
embeddings = np.array([w2vec.wv[aid] for aid in w2vec.wv.index_to_key], dtype=np.float32)# 词的向量
d=w2vec.wv.vectors.shape[1]

In [13]:
embeddings

array([[-5.2558500e-01,  6.8589520e-01,  2.7857850e+00, ...,
        -7.6781720e-01, -8.8462770e-01,  2.5221190e-01],
       [-1.1955168e+00, -4.3625513e-01,  2.2897859e+00, ...,
        -3.1780228e-01, -5.1577824e-01, -1.8228772e-01],
       [ 1.2625860e-01, -2.8093040e-01,  6.7882627e-01, ...,
        -1.5253893e+00,  6.9974488e-01,  6.5560019e-01],
       ...,
       [-5.2786149e-02, -1.6699682e-01,  1.2155543e-01, ...,
         2.4510559e-03, -9.3265817e-02, -9.8694554e-03],
       [-6.4299054e-02, -8.4796965e-02,  7.2802544e-02, ...,
        -3.0140541e-02, -7.9948129e-03, -3.9247241e-02],
       [-4.5567513e-02, -1.5156919e-01,  1.4277853e-01, ...,
        -1.8010862e-04, -5.5121612e-02, -1.9292869e-02]],
      shape=(1855603, 32), dtype=float32)

In [14]:
from cuml.neighbors import NearestNeighbors
knn=NearestNeighbors(n_neighbors=21,metric='euclidean') # 最近邻查找，每个样本返回21个（包括自己）
knn.fit(embeddings) # 接受（n_samples,n_features）训练模型

NearestNeighbors()

In [15]:
_, aid_nns = knn.kneighbors(embeddings)

In [16]:
aid_nns = aid_nns[:, 1:] # 排除自身，注意结果中存储的都是从0开始的索引，后续需要映射
aid_nns

array([[    228,     733,      32, ...,     677,   25872,    3055],
       [   1531,     103,     251, ...,   31968,   55855,   66149],
       [    121,     185,    1648, ...,   97706,   94324,    3812],
       ...,
       [1678964, 1853321, 1852485, ..., 1718780, 1703952, 1776763],
       [1852401, 1754146, 1821410, ..., 1745324, 1816346, 1754785],
       [1761561, 1709709, 1810423, ..., 1793281, 1725444, 1324822]],
      shape=(1855603, 20))

In [17]:
top_aids = [[aid_list[i] for i in row] for row in aid_nns]
top_aids

[[399992,
  1125095,
  811371,
  1783610,
  447645,
  57315,
  944778,
  959548,
  1065416,
  1119263,
  620545,
  127864,
  1611581,
  1066416,
  1148071,
  1837490,
  150294,
  884993,
  626338,
  440558],
 [1765072,
  1551213,
  1562705,
  33343,
  803544,
  1814856,
  1764685,
  152547,
  160091,
  964149,
  943806,
  1267798,
  432815,
  1132530,
  1005087,
  1658802,
  708928,
  824487,
  1314241,
  727012],
 [659399,
  435253,
  1457542,
  612920,
  747280,
  814997,
  1558907,
  585576,
  903606,
  641790,
  1433443,
  899498,
  972180,
  1557927,
  1754057,
  233909,
  1854872,
  522405,
  437862,
  1446825],
 [832192,
  137514,
  1498443,
  329725,
  493104,
  1605870,
  1194834,
  651801,
  1436280,
  1469978,
  614363,
  576116,
  608348,
  1708326,
  1413049,
  1733943,
  549612,
  368495,
  670066,
  1663535],
 [493104,
  670066,
  1030009,
  576116,
  823143,
  1634780,
  982423,
  536184,
  1708326,
  1498443,
  549612,
  29735,
  614363,
  832192,
  479834,
  177958,
 

In [18]:
sub = []
for idx, row in enumerate(top_aids):
    aid_x_real = aid_list[idx]  # 映射回真实 aid
    for aid_y in row:
        sub.append([aid_x_real, aid_y])

sub = pd.DataFrame(sub, columns=['aid_x','aid_y'])
sub.to_parquet('/home/mingyu/Recommand-System/项目/OTTO/save/co_visitation_result/word2vec.parquet', index=False)